# Continuous-focusing lattice 

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from orbit.core.bunch import Bunch
from orbit.core.bunch import BunchTwissAnalysis
from orbit.core.spacecharge import SpaceChargeCalc2p5D
from orbit.bunch_utils import collect_bunch
from orbit.envelope import Envelope
from orbit.envelope import EnvelopeTracker
from orbit.space_charge.sc2p5d import SC2p5D_AccNode
from orbit.space_charge.sc2p5d import setSC2p5DAccNodes
from orbit.teapot import ContinuousLinearFocusingTEAPOT
from orbit.teapot import TEAPOT_Lattice
from orbit.utils.consts import mass_proton

plt.style.use("../style.mplstyle")

In [ ]:
def cf_eq_radius(perveance: float, emittance: float, omega_0: float) -> float:
    if perveance == 0:
        return cf_eq_radius_emitt_dom(emittance, omega_0)
        
    return np.sqrt(perveance / omega_0**2) * np.sqrt(
        0.5 + 0.5 * np.sqrt(1.0 + (4.0 * omega_0**2 * emittance**2) / (perveance**2))
    )

def cf_eq_radius_emitt_dom(emittance: float, omega_0: float) -> float:
    return np.sqrt(emittance / omega_0)

def cf_eq_radius_sc_dom(perveance: float, omega_0: float) -> float:
    return np.sqrt(perveance) / omega_0

In [ ]:
def track_env_tbt(envelope: Envelope, tracker: EnvelopeTracker, turns: int, copy: bool = True) -> dict[str, np.ndarray]:    
    history_keys = ["rms_x", "rms_y"]
    history = {}
    for key in history_keys:
        history[key] = np.zeros(turns + 1)

    if copy:
        envelope_out = envelope.copy()
    else:
        envelope_out = envelope
        
    for i in range(turns + 1):
        if i > 0:
            tracker.track(envelope_out)
            
        cov_matrix = envelope_out.cov_matrix
        history["rms_x"][i] = 1000.0 * np.sqrt(cov_matrix[0, 0])
        history["rms_y"][i] = 1000.0 * np.sqrt(cov_matrix[2, 2])

    history["r"] = np.sqrt(2.0) * np.sqrt(history["rms_x"]**2 + history["rms_y"]**2)
    return history

## Matched envelope calculation

In [ ]:
length = 0.25
sigma_0 = math.radians(80.0)
omega_0 = sigma_0 * length

node = ContinuousLinearFocusingTEAPOT(
    length=length,
    nparts=20,
    kq=(omega_0**2),
)
lattice = TEAPOT_Lattice()
lattice.addNode(node)
lattice.initialize()

tracker = EnvelopeTracker(lattice, space_charge="2d")

In [ ]:
emittance = 0.0
perveance = 1e-6
bunch_length = 20.0

bunch_radius = cf_eq_radius(perveance=perveance, emittance=emittance, omega_0=omega_0)

cov_matrix = np.zeros((6, 6))
cov_matrix[0, 0] = cov_matrix[2, 2] = 0.25 * bunch_radius ** 2
cov_matrix[4, 4] = (bunch_length ** 2) / 12.0

envelope = Envelope(bunch=bunch, cov_matrix=cov_matrix)
envelope.set_intensity(
    perveance 
    * bunch_length
    * (envelope.beta()**2 * envelope.gamma()**3) 
    / (2.0 * envelope.classical_radius)
)

history = track_env_tbt(envelope, tracker, turns=100)

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(history["r"])
ax.set_ylim(0.0, np.mean(history["r"]) * 1.8)
ax.set_xlabel("Period")
ax.set_ylabel("Radius")